#P1 · Calidad y Limpieza de datos
## Dataset Airbnb Buenos Aires - Listings.csv.gz
**Fuente:** Inside Airbnb
**Fecha de Scrape:** Enero 2026
**Objetivo:** Identificar y corregir problemas de calidad para dejar el dataset listo para análisis

## Diagnóstico de calidad de datos

### Dimensiones

- Filas: 27.348
- Columnas: 85
- Duplicados: 0

### Problemas encontrados

1. **15 columnas con 100% de nulos** - eliminar
2. **Columnas 'license' (98% nulos) y 'host_about' (43% nulos)** - eliminar
3. **Columna 'review_scores_*' (12% nulos)** - corresponden a propiedades sin reseña, imputar con 0 o dejar como NaN según el análisis
4. **Columnas 'beds', 'bedrooms' y 'bathrooms' (<1% nulos)** - imputar con mediana

### Próximos pasos

- Eliminar columnas con >40% de nulos
- Imputar columnas críticas
- Convertir columnas de fecha a tipo datetime

**Limitación importante:** la columna "price" está 100% vacía en este scrape.
El dataset no permite análisis de precios.

### Insight destacado
847 propiedades (3.52%) llevan más de 3 años sin reseñas. Son listings activos, pero sin actividad real.


In [ ]:
import pandas as pd


df = pd.read_csv("../data/raw/listings.csv.gz")

print (df.dtypes)
print (df.shape)

In [ ]:
print(df["last_scraped"].unique())

In [ ]:
# Calcula el total de valores nulos por variable

df.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# Calcula el porcentaje de nulos por columna
# y muestra solo las que tienen al menos 1 nulo

nulos_pct = (df.isnull().sum()/len(df))*100
nulos_pct = nulos_pct.round(2)
nulos_pct = nulos_pct[nulos_pct > 0]
nulos_pct = nulos_pct.sort_values(ascending=False)
print(nulos_pct)

In [ ]:
# Cuántas filas duplicadas hay en el dataset

print(df.duplicated().sum())

# Ver las primeras 5 filas de las columnas más importantes

cols = ["id","name","room_type","beds","bedrooms","bathrooms"]
print(df[cols].head())

In [ ]:
# Columnas con más del 40% de nulos no aportan información útil
# Calcular el umbral y eliminar

umbral = 0.40
cols_a_eliminar = df.columns[df.isnull().mean()>umbral].tolist()

print(f"Columnas a eliminar: {len(cols_a_eliminar)}")
print(cols_a_eliminar)

In [ ]:
# Es raro que la columna price tenga tantos valores nulos. Se explora la variable

print(df["price"].head(20))


In [ ]:
# Probablemente airbnb no incluya el precio. Se busca si hay variables que contengan la variable price en el nombre

cols_precio = [cols for cols in df.columns if "price" in cols.lower()]
print(cols_precio)



In [ ]:
# Se elimina las columnas con más del 40% de nulos
# Inplace = False significa que no modifica el df directamente
# sino que devuelve un df nuevo llamado "data_clean"

df_clean = df.drop(columns=cols_a_eliminar)

print(f"Dimensiones originales: {df.shape}")
print (f"Dimensiones después: {df_clean.shape}")

In [ ]:
# Verificar los nulos que quedan en df df_clean

nulos_restantes = (df_clean.isnull().sum()/len(df_clean))*100
nulos_restantes = nulos_restantes.round(2)
nulos_restantes = nulos_restantes[nulos_restantes>0]
nulos_restantes = nulos_restantes.sort_values(ascending=False)

print(nulos_restantes)

In [ ]:
# Tenemos 3 grupos: variables >15%, 15%< y >3% y variables 2%<
# Del primer grupo le imputamos valor binario, de la segunda podemos prescindir o imputarle NaN y de la tercera imputación por grupos
# Ver qué valores tiene la variable room_type para imputar por grupos

print(df_clean["room_type"].value_counts())

In [ ]:
# Mediana de beds, bedrooms y bathrooms agrupado por room_type
cols_imputar = ["beds", "bedrooms", "bathrooms"]

print(df_clean.groupby("room_type")[cols_imputar].median())

In [ ]:
# Es raro que el valor hotel room tenga mediana = 0 en beds
# Ver cuántos hotel room tienen beds valor 0 vs valores reales

print(df_clean[df_clean["room_type"] == "Hotel room"]["beds"].value_counts(dropna=False))

In [ ]:
# Imputamos beds, bedrooms y bathrooms por mediana de room_type
# pero solo para los tipos que no son Hotel Room

tipos_a_imputar = ["Entire home/apt","Private room","Shared room"]

for col in ["beds","bedrooms","bathrooms"]:
    for tipo in tipos_a_imputar:
        mediana = df_clean.loc[df_clean["room_type"] == tipo, col].median()
        mask = (df_clean["room_type"] == tipo) & (df_clean[col].isnull())
        df_clean.loc[mask,col] = mediana

print("Nulos restantes en beds, bedrooms y bathrooms:")
print(df_clean[["beds","bedrooms","bathrooms"]].isnull().sum())


In [ ]:
# Ahora, convertir columnas de fecha a tipo datetime
# Ver qué columnas de fecha tenemos

cols_fecha = ["last_scraped","first_review","last_review"]

print(df_clean[cols_fecha].dtypes)
print(df_clean[cols_fecha].head())

In [ ]:
for col in cols_fecha:
    df_clean[col] = pd.to_datetime(df_clean[col])

print(df_clean[col].dtypes)

In [ ]:
# Creamos una variable binaria que indica si la propiedad tiene reseña o no
# notna() devuelve TRUE si el valor no es nulo, FALSE si el valor es nulo
# astype(int) convierte TRUE -> 1 y FALSE -> 0

df_clean["tiene_resenas"] = df_clean["review_scores_rating"].notna().astype(int)

print(df_clean["tiene_resenas"].value_counts())

In [ ]:
# En relación con las otras variables relacionadas con review podemos derivar nuevas variables:
# first_review + last_review → dias_activo (rango temporal)
# last_review + last_scraped → dias_desde_ultima_resena (qué tan reciente es la actividad)
# first_review + last_scraped → antiguedad_total (cuánto lleva en la plataforma)

print(df_clean[["first_review","last_review","last_scraped"]].describe())

In [ ]:
# dias_activo: rango entre primera y última celda
# solo tiene sentido para propiedades con reseñas

df_clean["dias_activo"] = (
    df_clean["last_review"] - df_clean["first_review"]
).dt.days

#días_desde_ultima_resena: qué tan reciente es la actividad
df_clean["dias_desde_ultima_resena"] = (
    df_clean["last_scraped"] - df_clean["last_review"]
).dt.days

#antiguedad_total: cuánto lleva en la plataforma desde primera reseña
df_clean["antiguedad_total"] = (
    df_clean["last_scraped"] - df_clean["first_review"]
).dt.days

print(df_clean[["dias_activo","dias_desde_ultima_resena","antiguedad_total"]].describe())

In [ ]:
# Cuántas propiedades llevan más de 3 años sin recibir una reseña

sin_actividad = (df_clean["dias_desde_ultima_resena"] > 365 * 3).sum()
total_con_resenas = df_clean["tiene_resenas"].sum()

print(f"Propiedades sin reseña hace más de 3 años: {sin_actividad}")
print(f"Como % del total con reseñas: {round(sin_actividad/total_con_resenas*100,2)}%")

In [ ]:
# Guardamos el dataset limpio en data/processed/

df_clean.to_csv("../data/processed/listings_clean.csv", index=False)

print(f"Dataset guardado: {df_clean.shape[0]:,} filas x {df_clean.shape[1]} columnas")

In [ ]:
# Creamos un resumen comparativo antes y después de la limpieza
reporte = {
    "filas": {"antes": df.shape[0], "después": df_clean.shape[0]},
    "columnas": {"antes": df.shape[1], "después": df_clean.shape[1]},
    "duplicados": {"antes":df.duplicated().sum(), "después": df_clean.duplicated().sum()},
    "nulos_totales": {"antes":df.isnull().sum().sum(), "después": df_clean.isnull().sum().sum()},
                      "variables_nuevas": ["tiene_resenas", "dias_activo", "dias_desde_ultima_resena", "antiguedad_total"]
        }

for key,val in reporte.items():
    if key == "variables_nuevas":
        print(f"{key}: {val}")
    else:
        print(f"{key}: {val['antes']} -> {val['después']}")

In [ ]:
# Guardamos elr eporte en reports/quality_report.md

reporte_md = f"""# Reporte de calidad de datos
##Dataset: Airbnb Buenos Aires - enero 2026

## Resumen antes/después

| Métrica | Antes | Después |
|---|---|---|
| Filas | {df.shape[0]:,} | {df_clean.shape[0]:,} |
| Columnas | {df.shape[1]} | {df_clean.shape[1]} |
| Duplicados | {df.duplicated().sum()} | {df_clean.duplicated().sum()} |
| Nulos totales | {df.isnull().sum().sum():,} | {df_clean.isnull().sum().sum():,} |
| Reducción de nulos | | 89.7% |

## Variables nuevas creadas
- 'tiene_resenas' - binaria (1/0)
- 'dias_activo' - días entre primera y última reseña
- 'dias_desde_ultima_resena' - días desde primera reseña al scrape

## Decisiones de limpieza
1. Eliminadas 17 columnas con más del 40% de nulos
2. Imputados 'beds', 'bedrooms', 'bathrooms' con mediana por 'room_type'
3. 'review_scores_*' conservados como NaN (propiedades sin reseñas)
4. Fechas convertidas a datetime

## Insight destacado
847 propiedades (3.52%) llevan más de 3 años sin reseña
"""

with open("../reports/quality_report.md","w", encoding="utf-8") as f:
    f.write(reporte_md)

print("Reporte guardado en reports/quality_report.md")


In [ ]:
# Verificamos que el archivo se creó correctamente

with open("../reports/quality_report.md", "r", encoding="utf-8") as f:
    print(f.read())